# 04. Evaluate runs and compare them

1. `src/evaluate.py` runs one Ultralytics validation pass per run (imgsz 640, batch 16, conf 0.001, iou 0.7, the locked call) and writes `results/<run>/val_metrics.csv`: per class images, boxes, precision, recall, F1, AP50, AP50-95, recall at precision 0.5 (R@P0.5), and bootstrap 95% intervals for egg and larva AP50. It also saves the PR curves and confusion matrices (at conf 0.25, readable, and at 0.001, Ultralytics' own).
2. `src/compare.py` puts the runs side by side: the AP50 and recall tables, the success rule check against R00, and the best run on val.

Use `SPLIT = "val"` all week. The plan uses `"test"` once, on Saturday, for all finished runs, and nothing changes after that.

## Parameters

In [ ]:
SPLIT    = "val"
RUNS     = ["R00-base", "R01-mixup", "R02-rfs", "R03-clspw"]   # runs without weights are skipped
BASELINE = "R00-base"

NDJSON       = "../data/kara-agro-ai-maize.ndjson"
SPLIT_CSV    = "../splits/split_v1.csv"
DRIVE_IMAGES = "/content/drive/Shared drives/<team drive>/kara-agro/images_1280"
RUNS_DIR     = "/content/drive/Shared drives/<team drive>/kara-agro/runs"
YOLO_DIR     = "/content/yolo"
RESULTS      = "../results"
SEED = 42

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U ultralytics
import os, sys
sys.path.append("..")
from src import make_yolo_dataset
if not os.path.exists(f"{YOLO_DIR}/dataset.yaml"):
    make_yolo_dataset.run(SPLIT_CSV, DRIVE_IMAGES, NDJSON, YOLO_DIR, neg_fraction=1.0, seed=SEED)
DATA = f"{YOLO_DIR}/dataset.yaml"     # plain dataset for every run, R02 included

## 1. Evaluate each run

Each run takes a few minutes on a T4 (one validation pass plus 1,000 bootstrap resamples). The last line of each run must say `[PASS] AP50 recomputed ... equals Ultralytics`; if it says FAIL, stop and check before using the extra metrics.

In [ ]:
from src import evaluate
done = []
for run in RUNS:
    w = f"{RUNS_DIR}/{run}/weights/best.pt"
    if not os.path.exists(w):
        print(f"skip {run}: no weights at {w}")
        continue
    print(f"\n===== {run} =====")
    evaluate.evaluate(run, w, DATA, SPLIT, f"{RESULTS}/{run}", seed=SEED)
    done.append(run)

## 2. Compare

In [ ]:
RUNS_STR = " ".join(RUNS)
!cd .. && python -m src.compare --results results --split {SPLIT} --runs {RUNS_STR} --baseline {BASELINE}
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/comparison_{SPLIT}.md").read()))

Paste the tables into the Run tracker tab of the Google Doc. The confusion matrices are in `results/<run>/plots_<split>/` (readable one under `cm_conf0.25/`). Commit `results/<run>/` on the run's `exp/` branch; the `.npz` file is small and keeps the per-image matches for the error review and paired comparisons.